<a href="https://colab.research.google.com/github/netsetos/agentic-ai-weekend-gcp-learners/blob/main/module-04-rag/lesson-4.8-live-evals/notebooks/GCP_Capstone_4.8_LiveEvals.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 4.8 Evals on the Live Lane
**Netsetos GenAI Engineering — GCP Capstone** · Module 4 · new in v2.0

Lesson 4.7 built the stopwatch in a notebook. This notebook runs it against the **deployed** lane - the lean profile of the Module 12 kit (`make up`), on the real corpus - as the two identities the gate needs, and then reads the result properly:

- **the scoreboard** - the gate's five rates, computed from a table with one record per golden row
- **the miss list** - every row that cost a point, classified: retrieval, generation, contract or plumbing
- **forensics** - the API's own retrieval, step for step from outside it, to see *where* a figure fell out
- **the corpus as it landed** - the ingest claims, chunk counts per tenant, the dead-letter queue, the roster
- **a ledger** - one line per round of the fix loop, so the room has a scoreboard

It needs a lane that is up (`make up`, `make ingest-corpus`, `make smoke` 3/3) and an account in `ADMIN_EMAILS` (so `make operators` let you mint tokens). Placeholders only: `documind-ai-YOUR-ID`. *Facts verified 2026-09-07 against deploy/evals/run_eval.py, rag-api and the ingest worker.*

## Setup — the lane, the golden set, and where the API lives
The API's URL is deterministic: the service name and the project **number**. The golden set is the committed one (59 rows: 31 lookup, 11 join, 7 refusal, 10 isolation) - the same file `make eval-live` and 12.7's pipeline read.

In [ ]:
!pip install -q google-genai==2.21.0 google-cloud-firestore==2.30.0 google-cloud-pubsub==2.40.0 google-cloud-discoveryengine==0.13.11 rank-bm25==0.2.2 pandas==2.3.2

from google.colab import auth
auth.authenticate_user()             # Application Default Credentials - and gcloud - for this session

PROJECT_ID = "documind-ai-YOUR-ID"   # CHANGE THIS: the project `make up` deployed the lane into
REGION     = "us-central1"           # where the services run (deploy/Makefile: REGION)
KIT        = "/content/agentic-ai-weekend-gcp-learners"   # the public repo: golden.jsonl and run_eval.py
BRANCH     = "main"   # the learner repo's branch: the notebooks and the kit (deploy/) ship there together

import json, os, subprocess, sys, time
import pandas as pd
import requests
import google.auth
from google.auth.transport.requests import AuthorizedSession, Request

# Cloud Run's URL is deterministic: the service name and the project NUMBER, no hash to look
# up - the same rule eventarc.tf and commands/lesson-12.2.sh build SELF_URL with.
creds, _ = google.auth.default()
NUMBER = AuthorizedSession(creds).get(
    f"https://cloudresourcemanager.googleapis.com/v1/projects/{PROJECT_ID}").json()["projectNumber"]
API = f"https://documind-api-{NUMBER}.{REGION}.run.app"
print("API:", API)

if not os.path.isdir(KIT):
    subprocess.run(["git", "clone", "--depth", "1", "-q", "-b", BRANCH,
                    "https://github.com/netsetos/agentic-ai-weekend-gcp-learners", KIT], check=True)
GOLDEN = [json.loads(l) for l in open(f"{KIT}/deploy/evals/golden.jsonl", encoding="utf-8") if l.strip()]
sys.path.insert(0, f"{KIT}/deploy/evals")
from run_eval import normalise          # the gate's own: number words to digits, separators out (F22)
print(len(GOLDEN), "golden rows:",
      ", ".join(f"{s}={sum(1 for r in GOLDEN if r['shape'] == s)}" for s in ("lookup", "join", "refusal", "isolation", "version")))

## Cell 1: The two identities
The gate needs a **member** (the UI's service account, on all three golden tenants' rosters) and an **outsider** (the chat service's account: may invoke the API, on no roster). Three things are not optional when minting: *who* (impersonate), *for which URL* (audience), *by what name* (include the e-mail). Finding F10 is the grant; F11 is the e-mail.

In [ ]:
from google.auth import impersonated_credentials

SCOPE = ["https://www.googleapis.com/auth/cloud-platform"]

def id_token_as(service_account: str, audience: str, include_email: bool = True) -> str:
    """A Google ID token minted AS a service account, for one audience.

    The same thing `gcloud auth print-identity-token --include-email
    --impersonate-service-account=... --audiences=...` does in Cloud Shell. Two things are not
    optional. The AUDIENCE: Cloud Run checks the token was minted for its URL, and the API's
    bearer leg (shared/iap.py) checks it against SELF_URL. The EMAIL: without include_email the
    token names no account, and the API refuses a caller it cannot look up on a roster -
    "bearer token carries no verified email", finding F11 of the first live run."""
    source, _ = google.auth.default()
    target = impersonated_credentials.Credentials(
        source_credentials=source, target_principal=service_account, target_scopes=SCOPE)
    idc = impersonated_credentials.IDTokenCredentials(
        target, target_audience=audience, include_email=include_email)
    idc.refresh(Request())
    return idc.token

MEMBER_SA   = f"documind-ui-sa@{PROJECT_ID}.iam.gserviceaccount.com"    # on acme, zeta AND globex (make roster)
OUTSIDER_SA = f"documind-outsider-sa@{PROJECT_ID}.iam.gserviceaccount.com"  # may invoke the API; on no roster

# Minting either needs roles/iam.serviceAccountTokenCreator on that account, for YOU.
# `make operators` grants it to ADMIN_EMAILS; Project Owner does not include it (finding F10).
MEMBER   = id_token_as(MEMBER_SA, API)
OUTSIDER = id_token_as(OUTSIDER_SA, API)
print("member   token:", MEMBER[:20], "... minted as", MEMBER_SA)
print("outsider token:", OUTSIDER[:20], "... minted as", OUTSIDER_SA)

### One question, read properly
`answerable`, `confidence`, the citations with their pages - the fields the gate scores. lk-16 is in every tenant's corpus, which is why the smoke test asks it too.

In [ ]:
def ask(question: str, tenant: str, token: str, top_k: int = 6, **extra) -> tuple:
    """POST /v1/query the way run_eval.py does. Returns (status, body).

    top_k is what the API reranks DOWN to before the model sees anything (main.py:
    retrieve -> rerank(req.top_k) -> generate); the gate sends 6. A failed call returns
    the status and whatever the service said, so a 401 is never mistaken for a refusal."""
    body = {"query": question, "tenant_id": tenant, "top_k": top_k, "stream": False, **extra}
    r = requests.post(f"{API}/v1/query", json=body, timeout=90,
                      headers={"Authorization": f"Bearer {token}"})
    try:
        return r.status_code, r.json()
    except ValueError:
        return r.status_code, {"error": r.text[:300]}

# lk-16: the Payment of Gratuity Act, 1972 - in every tenant's corpus, so the smoke test asks it too.
status, body = ask("After how many years of continuous service does gratuity become payable?", "acme", MEMBER)
print(status, "| answerable =", body.get("answerable"), "| confidence =", body.get("confidence"),
      "| citations =", len(body.get("citations") or []))
print((body.get("answer") or body)[:400])
for c in body.get("citations") or []:
    print("  -", c["source_uri"].rsplit("/", 1)[-1], "p.", c.get("page"), "|", c["quote"][:80])

## Cell 2: Four status codes
200, 401, 403 and 422 are four different questions about *who is asking* and *what was sent*. A gate that lumps them together scores an outage as a refusal. Three of the four are induced on purpose here.

In [ ]:
# Four status codes, four different questions about WHO is asking and WHAT was sent. Read them
# before you read an answer: a gate that cannot tell them apart scores an outage as a refusal.
NO_EMAIL = id_token_as(MEMBER_SA, API, include_email=False)     # finding F11, re-induced on purpose
Q = "What notice period applies during probation?"

probes = [
    ("200  the member, a well-formed request",        lambda: ask(Q, "acme", MEMBER)),
    ("401  a token that names no email (F11)",        lambda: ask(Q, "acme", NO_EMAIL)),
    ("403  the outsider: may invoke, on no roster",   lambda: ask(Q, "acme", OUTSIDER)),
    ("422  a body with no tenant_id (F15's shape)",   lambda: ask(Q, "", MEMBER)),
]
for label, probe in probes:
    status, body = probe()
    detail = body.get("detail") or body.get("error") or ""
    if isinstance(detail, list):                       # pydantic's 422 lists the missing fields
        detail = "; ".join(f"{'.'.join(map(str, d.get('loc', [])))}: {d.get('msg')}" for d in detail)
    print(f"  expect {label:46} got {status}  {str(detail)[:70]}")

## Cell 3: The scoreboard — every row, both identities
Five to six minutes: fifty-nine questions as the member, the ten isolation rows again as the outsider. The five rates use the gate's denominators (all rows of that kind, so a 401 costs the same point a refusal would). `cited_anchors` is the closest honest thing to recall - a lower bound, printed, never thresholded.

In [ ]:
THRESHOLDS = {"answerable_rate": 0.80, "citation_rate": 0.95, "must_contain_rate": 0.85,
              "refusal_rate": 0.90, "isolation_403_rate": 1.00}     # deploy/evals/run_eval.py, verbatim

def run_golden(rows: list, member: str, outsider: str) -> pd.DataFrame:
    """Every row as the member; the isolation rows once more as the outsider.

    One record per row with what the API said. The five numbers are computed FROM this
    table, so a lost point is always traceable to a row - the thing the first live run's
    five percentages could not tell you."""
    out = []
    for i, r in enumerate(rows, 1):
        status, body = ask(r["question"], r["tenant"], member)
        ok = status == 200
        text = normalise(body.get("answer") or "") if ok else ""      # the gate's normalise(): "twelve weeks" == "12 weeks"
        cites = (body.get("citations") or []) if ok else []
        rec = {"id": r["id"], "shape": r["shape"], "tenant": r["tenant"], "status": status,
               "expect_answer": r["answerable"],
               "answerable": bool(body.get("answerable")) if ok else None,
               "confidence": body.get("confidence") if ok else None,
               "cites": len(cites),
               "has_all": ok and all(normalise(w) in text for w in r.get("must_contain", [])),
               "leak": any(normalise(n) in text for n in r.get("must_not_contain", [])),
               # A LOWER BOUND on retrieval recall: anchors the model CITED, by source file or
               # quote. /v1/query returns citations, not the retrieved set, so recall itself is
               # not observable from outside the service (12.7 says so); this is the nearest thing.
               "cited_anchors": sum(1 for a in r.get("must_retrieve", [])
                                    if any(a.lower() in (c.get("source_uri", "") + " " + c.get("quote", "")).lower()
                                           for c in cites)),
               "anchors": len(r.get("must_retrieve", [])),
               "answer": (body.get("answer") or "")[:160] if ok else str(body)[:160],
               "question": r["question"]}
        if r["shape"] == "isolation":
            rec["outsider_status"], _ = ask(r["question"], r["tenant"], outsider)
        out.append(rec)
        if i % 10 == 0:
            print(f"  {i}/{len(rows)} rows asked")
        time.sleep(0.2)
    return pd.DataFrame(out)

def scores(df: pd.DataFrame) -> dict:
    """The gate's five rates, the gate's way: denominators are ALL the rows of that kind, so a
    row that failed with a 401 costs the same point a refusal would."""
    ok = df[df.status == 200]
    answered = ok[ok.expect_answer & (ok.answerable == True)]
    unanswerable = ok[~ok.expect_answer]
    iso = df[df["shape"] == "isolation"]
    n = max(int(df.expect_answer.sum()), 1)
    u = max(int((~df.expect_answer).sum()), 1)
    return {"answerable_rate": len(answered) / n,
            "citation_rate": float((answered.cites > 0).mean()) if len(answered) else 0.0,
            "must_contain_rate": float(answered.has_all.mean()) if len(answered) else 0.0,
            "refusal_rate": int((unanswerable.answerable == False).sum()) / u,
            "isolation_403_rate": float((iso.outsider_status == 403).mean()) if len(iso) else 0.0}

RESULTS = run_golden(GOLDEN, MEMBER, OUTSIDER)
SCORES = scores(RESULTS)
print()
for k, v in SCORES.items():
    print(f"  [{'PASS' if v >= THRESHOLDS[k] else 'FAIL'}] {k:20} {v:6.1%}  (threshold {THRESHOLDS[k]:.0%})")
print(f"\n  must_not_contain hits: {int(RESULTS.leak.sum())}   (a stop-everything, whatever the five say)")

## Cell 4: The miss list
The gate prints *what* happened to every row that cost a point. This names *where to look*: `cites=0` on a refusal is retrieval, `cites>0` is generation, a wrong figure is the contract, a status code is plumbing.

In [ ]:
def classify(rec) -> str:
    """Where a lost point points. The gate prints WHAT happened; this names WHERE to look."""
    if rec.status != 200:
        return {401: "plumbing: identity (audience, email)", 403: "plumbing: roster",
                422: "plumbing: request body"}.get(rec.status, f"plumbing: HTTP {rec.status}")
    if rec.expect_answer and not rec.answerable:
        return ("retrieval: nothing relevant reached the model" if rec.cites == 0
                else "generation: the model saw a source and still declined")
    if rec.expect_answer and not rec.has_all:
        return "contract: answered, but not with the row's figure"
    if not rec.expect_answer and rec.answerable:
        return "over-answering: it should have refused"
    return "ok"

RESULTS["cause"] = [classify(r) for r in RESULTS.itertuples()]
MISSES = RESULTS[RESULTS.cause != "ok"]
print(f"rows that cost a point: {len(MISSES)} of {len(RESULTS)}\n")
print(MISSES.cause.value_counts().to_string(), "\n")
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 56)
print(MISSES[["id", "shape", "tenant", "status", "answerable", "confidence", "cites",
              "cited_anchors", "anchors", "cause", "question"]].to_string(index=False))

## Cell 5: Retrieval forensics — where did the figure go?
The API's retrieval, step for step, from outside it: the query embedding on the **regional** endpoint, Firestore's `find_nearest` with the tenant filter (top 20), the Ranking API over those twenty (`top_k`). Three verdicts, three rungs of the ladder: RETRIEVAL (chunking or embedding), RANKING (the window), GENERATION (the prompt, or the row's phrasing).

In [ ]:
from google import genai
from google.genai import types
from google.cloud import firestore
from google.cloud.firestore_v1.base_query import FieldFilter
from google.cloud.firestore_v1.vector import Vector
from google.cloud.firestore_v1.base_vector_query import DistanceMeasure
from google.cloud import discoveryengine_v1 as discoveryengine

# The API's own retrieval, step for step, from OUTSIDE it: the query embedding (regional -
# embeddings are not served from the global endpoint), Firestore's find_nearest with the tenant
# filter (the lean profile's entire retriever), then the Ranking API over those twenty. Run it
# for a row that was refused and see WHERE the row's figure fell out: never retrieved, retrieved
# but ranked out of the top_k window, or inside the window and still declined.
embed_client = genai.Client(enterprise=True, project=PROJECT_ID, location=REGION)
db = firestore.Client(project=PROJECT_ID, database="(default)")
ranker = discoveryengine.RankServiceClient()

def retrieve_like_the_api(question: str, tenant: str, top_k_retrieve: int = 20) -> list:
    vec = embed_client.models.embed_content(
        model="text-embedding-005", contents=question,
        config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY", output_dimensionality=768)
    ).embeddings[0].values
    hits = (db.collection("chunks").where(filter=FieldFilter("tenant_id", "==", tenant))
            .find_nearest("embedding", Vector(vec), distance_measure=DistanceMeasure.COSINE,
                          limit=top_k_retrieve, distance_result_field="d").get())
    out = []
    for h in hits:
        d = h.to_dict()
        out.append({"id": h.id, "source": d["source_uri"].rsplit("/", 1)[-1], "page": d.get("page_start"),
                    "vec": round(1.0 - d.get("d", 1.0), 4), "text": d["text"]})
    return out

def rerank_like_the_api(question: str, chunks: list, top_k: int = 6) -> list:
    cfg = ranker.ranking_config_path(project=PROJECT_ID, location="global",
                                     ranking_config="default_ranking_config")
    resp = ranker.rank(request=discoveryengine.RankRequest(
        ranking_config=cfg, model="semantic-ranker-fast-004", top_n=top_k, query=question,
        records=[discoveryengine.RankingRecord(id=str(i), content=c["text"]) for i, c in enumerate(chunks)]))
    return [dict(chunks[int(r.id)], rerank=round(r.score, 3)) for r in resp.records]

def where_did_it_go(row_id: str, top_k: int = 6) -> None:
    row = next(r for r in GOLDEN if r["id"] == row_id)
    wants = [w.lower() for w in row.get("must_contain", [])]
    holds = lambda c: any(w in c["text"].lower() for w in wants)
    cands = retrieve_like_the_api(row["question"], row["tenant"])
    top = rerank_like_the_api(row["question"], cands, top_k)
    print(f"{row_id}: {row['question']}")
    print(f"  must_contain {row.get('must_contain')} | tenant {row['tenant']}")
    print(f"  vector top-20   : figure at rank {[i for i, c in enumerate(cands, 1) if holds(c)] or 'NONE'}")
    print(f"  reranked top-{top_k}  : figure at rank {[i for i, c in enumerate(top, 1) if holds(c)] or 'NONE'}")
    for i, c in enumerate(top, 1):
        print(f"   {i}. {'*' if holds(c) else ' '} {c['source']:42} p.{str(c['page']):4} rerank={c['rerank']:.3f} vec={c['vec']:.4f}")
    if not any(holds(c) for c in cands):
        print("  -> RETRIEVAL: the figure is in none of the twenty. Chunking or embedding - not the prompt.")
    elif not any(holds(c) for c in top):
        print(f"  -> RANKING: retrieved, then dropped by the reranker or the top_k window ({top_k}).")
    else:
        print("  -> GENERATION: the figure was in the window the model saw. The prompt, or the row's phrasing.")

# A refused row from the miss list, or any golden id you like.
refused = MISSES[MISSES.cause.str.startswith(("retrieval", "generation"))].id.tolist()
where_did_it_go(refused[0] if refused else "lk-16")

## Cell 6: The corpus, as it landed — claims, counts, the DLQ
Every upload is one claim in `documents`: `processing`, `indexed`, or `failed` with the error the worker hit (F17, F18 and F19 were read from that field). The dead-letter queue holds what failed twelve deliveries (F16). Pulled without acking.

In [ ]:
from collections import Counter
from google.cloud import firestore, pubsub_v1
from google.cloud.firestore_v1.base_query import FieldFilter

db = firestore.Client(project=PROJECT_ID, database="(default)")

# 1. The claims. Every upload is one document in `documents`: processing, indexed, or failed
#    with the error the worker hit. Findings F17, F18 and F19 were read from this field.
claims = [dict(d.to_dict(), doc_key=d.id) for d in db.collection("documents").stream()]
print("claims by status:", dict(Counter(c.get("status") for c in claims)))
for c in claims:
    if c.get("status") == "failed":
        print(f"  FAILED {c['doc_key'][:12]}  {c.get('gcs_uri', '').rsplit('/', 1)[-1]:44} {c.get('error', '')[:100]}")

# 2. What actually landed, per tenant - the counts the isolation rows depend on.
for t in ("acme", "zeta", "globex"):
    n = db.collection("chunks").where(filter=FieldFilter("tenant_id", "==", t)).count().get()[0][0].value
    print(f"  chunks[{t:6}] = {int(n)}")

# 3. The dead-letter queue: messages that failed twelve deliveries (finding F16). Pulled WITHOUT
#    acking, so they stay for the operator; the attribute says how many times Pub/Sub tried.
sub = pubsub_v1.SubscriberClient()
path = sub.subscription_path(PROJECT_ID, "ingest-dlq-sub")
resp = sub.pull(request={"subscription": path, "max_messages": 10}, timeout=20)
print(f"  DLQ: {len(resp.received_messages)} message(s)")
for m in resp.received_messages:
    rec = json.loads(m.message.data)
    print(f"    {rec.get('name'):52} deliveries={m.message.attributes.get('CloudPubSubDeadLetterSourceDeliveryCount')}")
if resp.received_messages:      # let them go straight back - not acked, just no longer held here
    sub.modify_ack_deadline(request={"subscription": path, "ack_deadline_seconds": 0,
                                     "ack_ids": [m.ack_id for m in resp.received_messages]})

### The roster
Two readers, two index needs (12.8): the point lookup the API makes needs none; the reverse lookup the UI makes needs the collection-group index on `members.email` (F14).

In [ ]:
# The roster: tenants/{tenant}/members/{email}. Two readers, two index needs (12.8): the POINT
# lookup rag-api makes (is_member - one document read, no index) and the REVERSE lookup the UI
# makes (tenant_for - a collection-group query on members.email, which needs the index finding
# F14 added). A verified person on no roster is a 403, never a default tenant.
for t in ("acme", "zeta", "globex"):
    members = [d.id for d in db.collection("tenants").document(t).collection("members").stream()]
    print(f"  {t:7} {len(members)} member(s): {', '.join(members)[:110]}")

def tenant_for(email: str):
    try:
        hits = (db.collection_group("members")
                  .where(filter=FieldFilter("email", "==", email.lower())).limit(1).get())
        return next((d.reference.parent.parent.id for d in hits), None)
    except Exception as e:      # FailedPrecondition: the collection-group index is missing (F14)
        return f"F14 - {type(e).__name__}: {str(e)[:150]}"

print("\n  reverse lookup, the member identity :", tenant_for(MEMBER_SA))
print("  reverse lookup, the outsider identity:", tenant_for(OUTSIDER_SA))

## Cell 7: The fix loop's ledger
One change per round, the API alone redeployed, all five rates written down beside the baseline. A round that moves one rate by breaking another is not progress.

In [ ]:
RUNS = "runs.jsonl"      # one line per round, so the loop has a memory and the room has a scoreboard

def record(label: str, sc: dict) -> None:
    with open(RUNS, "a", encoding="utf-8") as f:
        f.write(json.dumps({"label": label, "when": time.strftime("%Y-%m-%d %H:%M"), **sc}) + "\n")

record("baseline", SCORES)
hist = pd.DataFrame([json.loads(l) for l in open(RUNS, encoding="utf-8")])
print(hist.to_string(index=False, float_format=lambda v: f"{v:.1%}"))

# After ONE change and a redeploy of the API alone
#   make build deploy-services PROJECT=... SERVICES_lean=api SCRIPTS_lean=commands/lesson-12.2.sh
# run the round again and write it down beside the baseline:
#   RESULTS = run_golden(GOLDEN, MEMBER, OUTSIDER); SCORES = scores(RESULTS)
#   record("round 1: <what you changed>", SCORES)

## Cell 8: The gate itself
The file 12.7 ships, unchanged, with the two tokens minted above. Its exit code is the only thing CI reads: 0 merges, 1 is a threshold, 2 is a leak.

In [ ]:
# The gate 12.7 ships, unchanged, against this deployment, with the two tokens minted above.
# Its exit code is the only thing CI reads: 0 merges, 1 is a threshold, 2 is a leak.
env = dict(os.environ, DOCUMIND_ID_TOKEN=MEMBER, DOCUMIND_OUTSIDER_TOKEN=OUTSIDER)
r = subprocess.run(["python", f"{KIT}/deploy/evals/run_eval.py", "--api-url", API],
                   env=env, capture_output=True, text=True)
print(r.stdout[-5000:])
print("exit code:", r.returncode)

## Cell 9: The eight knobs — an ablation harness, retrieval only
Every component of the lane is its own eval axis, with its own metric and its own failure mode: parsing, chunking, the embedding model, the retrieval strategy, the reranker, query processing, context assembly, the index and its filters. The lesson page's Part 5 gives the verdict on each from the six runs above — two are already thresholded, three are worth one measurement, three are not worth one yet.

This cell makes the three measurements in one pass, with no model in the loop: the lane's own path against the same path without its reranker, against a deeper candidate list, and against 4.5's hybrid leg (which this profile does not wire). Next to every reranked number it prints the recall at the candidate depth, because a reranker can only reorder what the retriever returned.

> One knob per arm, the golden set frozen, everything else held. Report the quality delta **and** the latency delta, or the number means nothing.

The same harness ships beside the gate as `deploy/evals/ablate.py` — `make ablate PROJECT=... ABLATE_ARGS="--limit 5"` from Cloud Shell is the wiring check, and without the limit it is the measurement (the lesson page's Part 5 has the session's commands).

**Measured on 8 September 2026** (43 rows, Cloud Shell): dense 5 alone 0.91 recall@5 / 0.81 MRR; the lane's dense 20 → rerank 5 **0.96 / 0.91**; dense 50 → rerank 5 0.99 / 0.94 at four times the latency; hybrid 20 → rerank 5 **identical to the lane's row** at alpha 0.5, and again at 0.7 on ACME's 36 rows. So: the reranker earns its place, 20 candidates is deep enough, and the sparse leg stays unwired on windows this size. Under each arm the cell now prints its miss list — which row lost a point, and whether the anchor was *ranked out* (the reranker's) or *not retrieved* (the retriever's, or the corpus's).

**The miss list, named** (the second pass, alpha 0.7): without the reranker the lane loses Form 16's date (lk-02), the CFO's threshold (lk-08), the contractor's USB drive (jn-07) and the ministry handbook's gratuity clause (jn-11) at k=5; the reranker brings three back from the twenty. The one anchor at rank 21–50 is jn-07's IT-SEC-04, and it is the **chunking** knob: the handbook's ten clauses fit in the worker's *first* 2,000-character window, so ten answers share one vector — one window on the lane, ten chunks in the kit's loader. jn-08's principal Act is *ranked out* on purpose (the amendment answers the question). Hybrid equals dense because no golden question carries a literal code — the shape the sparse leg exists for is missing from the set.


In [ ]:
# Cell 9: the ablation harness - retrieval only, no model in the loop, against the frozen golden set.
# One knob at a time, everything else held: the lane's own path (Firestore dense 20 -> Rank API 5),
# the same without the reranker, the reranker over 50 candidates, and 4.5's hybrid leg (dense + BM25
# fused by RRF at the kit's alpha=0.7) - which this profile does NOT wire (retriever.py falls back
# to find_nearest; hybrid.py rides Vector Search only). Scored on the 43 rows that carry anchors, by
# recall and MRR against source file + text. Next to every reranked number is the recall at the
# candidate depth, because a reranker can only reorder what the retriever returned: if recall@20
# did not move, no reranker setting will. Cost: ~170 query embeddings, ~130 Rank API requests,
# Firestore reads of three tenants' text - and not one generation call.
import re, statistics, time
from google import genai
from google.genai import types
from google.cloud import firestore
from google.cloud.firestore_v1.base_query import FieldFilter
from google.cloud.firestore_v1.vector import Vector
from google.cloud.firestore_v1.base_vector_query import DistanceMeasure
from google.cloud import discoveryengine_v1 as discoveryengine
from rank_bm25 import BM25Okapi

embed_client = genai.Client(enterprise=True, project=PROJECT_ID, location=REGION)   # embeddings: regional
db = firestore.Client(project=PROJECT_ID, database="(default)")
ranker = discoveryengine.RankServiceClient()
RANKING_CONFIG = ranker.ranking_config_path(project=PROJECT_ID, location="global", ranking_config="default_ranking_config")

def embed_query(q: str) -> list:
    return embed_client.models.embed_content(
        model="text-embedding-005", contents=q,
        config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY", output_dimensionality=768)).embeddings[0].values

def _tok(s: str) -> list:
    return re.findall(r"[a-z0-9\-]+", s.lower())

_TENANT = {}
def tenant_corpus(tenant: str):
    """One tenant's text chunks (no embeddings: 768 floats each) and a BM25 index over them - the sparse leg."""
    if tenant not in _TENANT:
        rows = []
        for d in (db.collection("chunks").where(filter=FieldFilter("tenant_id", "==", tenant))
                    .select(["text", "source_uri"]).stream()):
            x = d.to_dict()
            rows.append({"chunk_id": d.id, "text": x.get("text", ""), "source_uri": x.get("source_uri", "")})
        _TENANT[tenant] = (rows, {r["chunk_id"]: r for r in rows}, BM25Okapi([_tok(r["text"]) for r in rows]))
    return _TENANT[tenant]

def dense(question: str, tenant: str, k: int) -> list:
    """The lane's retriever on this profile: Firestore find_nearest with the tenant pre-filter."""
    docs = (db.collection("chunks").where(filter=FieldFilter("tenant_id", "==", tenant))
              .find_nearest(vector_field="embedding", query_vector=Vector(embed_query(question)),
                            distance_measure=DistanceMeasure.COSINE, limit=k, distance_result_field="d").get())
    out = []
    for d in docs:
        x = d.to_dict()
        out.append({"chunk_id": d.id, "text": x.get("text", ""), "source_uri": x.get("source_uri", "")})
    return out

def hybrid(question: str, tenant: str, k: int, alpha: float = 0.7) -> list:
    """4.5's RRF over the dense ids and a BM25 leg - the fusion hybrid.py runs on Vector Search, in-process."""
    rows, by_id, bm25 = tenant_corpus(tenant)
    d_ids = [c["chunk_id"] for c in dense(question, tenant, k)]
    scores = bm25.get_scores(_tok(question))
    s_ids = [rows[i]["chunk_id"] for i in sorted(range(len(rows)), key=lambda i: -scores[i])[:k] if scores[i] > 0]
    fused = {}
    for rank, cid in enumerate(d_ids):
        fused[cid] = fused.get(cid, 0.0) + alpha / (60 + rank + 1)
    for rank, cid in enumerate(s_ids):
        fused[cid] = fused.get(cid, 0.0) + (1 - alpha) / (60 + rank + 1)
    return [by_id[cid] for cid, _ in sorted(fused.items(), key=lambda kv: -kv[1])[:k] if cid in by_id]

def rerank(question: str, chunks: list, top_n: int = 5) -> list:
    if not chunks:
        return chunks
    records = [discoveryengine.RankingRecord(id=str(i), title=c["source_uri"].rsplit("/", 1)[-1], content=c["text"][:4000])
               for i, c in enumerate(chunks)]
    resp = ranker.rank(request=discoveryengine.RankRequest(
        ranking_config=RANKING_CONFIG, model="semantic-ranker-fast-004", top_n=top_n, query=question, records=records))
    return [chunks[int(r.id)] for r in resp.records]

def _hay(c: dict) -> str:
    return (c["source_uri"] + " " + c["text"]).lower()

def recall_at(chunks: list, anchors: list, k: int) -> float:
    hay = [_hay(c) for c in chunks[:k]]
    return sum(1 for a in anchors if any(a.lower() in h for h in hay)) / len(anchors)

def mrr(chunks: list, anchors: list) -> float:
    for i, c in enumerate(chunks, 1):
        if any(a.lower() in _hay(c) for a in anchors):
            return 1.0 / i
    return 0.0

ARMS = [  # name, candidates(question, tenant), reranked to 5?
    ("dense 5, no reranker",                      lambda q, t: dense(q, t, 5),   False),
    ("dense 20 -> rerank 5   (the lane)",         lambda q, t: dense(q, t, 20),  True),
    ("dense 50 -> rerank 5",                      lambda q, t: dense(q, t, 50),  True),
    ("hybrid 20 -> rerank 5  (4.5, not wired)",   lambda q, t: hybrid(q, t, 20), True),
]
ROWS = [r for r in GOLDEN if r.get("must_retrieve")]
print(f"{len(ROWS)} rows with anchors, {sum(len(r['must_retrieve']) for r in ROWS)} anchors, one knob per arm\n")
print(f"{'arm':44} {'recall@depth':>12} {'recall@5':>9} {'mrr':>6} {'rows@1.0':>8} {'p95 ms':>7}")
for name, cands, do_rerank in ARMS:
    r_depth, r5, rr, ms, errors, misses = [], [], [], [], [], []
    for row in ROWS:
        t0, c, top = time.time(), None, None
        for attempt in (1, 2):                      # one retry: a dropped stream is the network, not the lane
            try:
                c = cands(row["question"], row["tenant"])
                top = rerank(row["question"], c) if do_rerank else c[:5]
                break
            except Exception as e:                  # one row must not stop the arm
                c = None
                if attempt == 2:
                    errors.append(f"{row['id']}: {type(e).__name__}: {str(e)[:120]}")
                else:
                    time.sleep(2)
        if c is None:
            continue
        ms.append((time.time() - t0) * 1000)
        r_depth.append(recall_at(c, row["must_retrieve"], len(c)))     # the ceiling for this arm
        r5.append(recall_at(top, row["must_retrieve"], 5))
        rr.append(mrr(top, row["must_retrieve"]))
        if r5[-1] < 1.0:                            # the miss list: the rows that cost a point, and WHERE the anchor went
            got = [_hay(x) for x in top]
            missing = [a for a in row["must_retrieve"] if not any(a.lower() in h for h in got)]
            deeper = [a for a in missing if any(a.lower() in _hay(x) for x in c)]
            misses.append(f"{row['id']}: {', '.join(missing)} - "
                          + ("ranked out (in the candidates, not the five)" if deeper and len(deeper) == len(missing)
                             else f"not retrieved at depth {len(c)}" if not deeper
                             else f"{', '.join(deeper)} ranked out; the rest not retrieved at depth {len(c)}"))
    p95 = sorted(ms)[int(0.95 * (len(ms) - 1))]
    print(f"{name:44} {statistics.mean(r_depth):12.2f} {statistics.mean(r5):9.2f} {statistics.mean(rr):6.2f} "
          f"{sum(1 for x in r5 if x == 1.0):8d} {p95:7.0f}" + (f"   ({len(errors)} rows failed twice)" if errors else ""))
    for e in errors[:3]:
        print("      ", e)
    for m_ in misses:
        print("       lost:", m_)
print("\nRead it in this order: recall@depth is the reranker's ceiling - if the 20 and 50 rows agree, depth is not the knob;"
      "\nrecall@5 of the lane's row minus the first row is the reranker's lift, and p95 is what it costs;"
      "\nthe hybrid row says whether a BM25 leg is worth wiring - 20 of the anchors are clause codes, its home ground.")


### The same harness, as the kit's script
`deploy/evals/ablate.py` is Cell 9 with a command line: four arms, recall and MRR, a miss list, a ledger. `make ablate` runs it from Cloud Shell.


In [ ]:
# The harness above, as the kit's script: deploy/evals/ablate.py carries the same four arms, the same recall and MRR
# against source file + text, the same miss list per arm, and a --ledger that appends one JSON line per arm - the
# loop's memory. make ablate PROJECT=... runs it from Cloud Shell in about three minutes; --limit 5 is the wiring
# check. The arms and the options, read from the file rather than retyped; the measured run is the table below.
src = open(f"{KIT}/deploy/evals/ablate.py", encoding="utf-8").read()
print("\n".join(l for l in src.splitlines() if l.startswith("def ") or "lambda q, t:" in l))
print()
r = subprocess.run([sys.executable, f"{KIT}/deploy/evals/ablate.py", "--help"], capture_output=True, text=True)
print(r.stdout.strip() or r.stderr.strip())


## Cell 10: A document changes - red, then green
The ledger (12.5, `deploy/INDEXING.md`) makes a reindex a release. The handbook re-issued under its own name (NP-03: 60 to 90 days) is indexed with the previous version's chunks reused by hash, the old rows retired with an `expire_at`, the answer moved - and the gate, scoped to the rows that cite the handbook (`run_eval.py --source`), goes red on `lk-06` and `vr-01` until the rows move with the document. Then the undo: the original bytes again, reactivated without a re-embedding, and the same gate green. Nothing is deployed; the lane ends where it began.


In [ ]:
# A DOCUMENT CHANGES. Revision 2 of the handbook (evals/demo/hr_policy_2026_v2.md) goes in AS hr_policy_2026.md; the
# worker's line says what it cost (reused / embedded / retired); the gate scoped to the handbook's ten rows goes red
# on the rows whose figure moved; the original bytes bring revision 1 back (ingest_reactivated, nothing embedded), and
# the same gate is green. make reindex FILE= NAME= runs the same from Cloud Shell - the offline gate first, so a
# re-issue whose rows did not move stops before the upload.
import datetime
V1, V2 = f"{KIT}/deploy/evals/corpus/acme/hr_policy_2026.md", f"{KIT}/deploy/evals/demo/hr_policy_2026_v2.md"
OBJ = f"gs://{PROJECT_ID}-uploads/acme/hr_policy_2026.md"
env_g = dict(os.environ, DOCUMIND_ID_TOKEN=MEMBER, DOCUMIND_OUTSIDER_TOKEN=OUTSIDER)

def wait_for(events, since, minutes=5):
    """The worker's line for this upload, or None: Cloud Logging every 10 s, the way make reindex waits."""
    ev = " OR ".join(f'jsonPayload.event="{e}"' for e in events)
    q = (f'resource.type="cloud_run_revision" AND resource.labels.service_name="documind-ingest" AND ({ev}) '
         f'AND jsonPayload.tenant="acme" AND timestamp>="{since.strftime("%Y-%m-%dT%H:%M:%SZ")}"')
    for _ in range(minutes * 6):
        r = subprocess.run(["gcloud", "logging", "read", q, "--project", PROJECT_ID, "--limit", "1", "--format=json"],
                           capture_output=True, text=True)
        rows = json.loads(r.stdout or "[]")
        if rows:
            return rows[0]["jsonPayload"]
        time.sleep(10)

def upload(path):
    subprocess.run(["gcloud", "storage", "cp", path, OBJ, "--project", PROJECT_ID], check=True, capture_output=True)

def scoped_gate():
    r = subprocess.run(["python", f"{KIT}/deploy/evals/run_eval.py", "--api-url", API, "--source", "hr_policy_2026.md"],
                       env=env_g, capture_output=True, text=True)
    print("\n".join(l for l in r.stdout.splitlines() if l.strip().startswith(("[", "vr-", "lk-", "Blocked", "All", "A version", "scoped"))))
    return r.returncode

since = datetime.datetime.now(datetime.timezone.utc)
upload(V2)
line = wait_for(("ingest_ok", "ingest_reactivated"), since)
assert line, "no worker line in 5 min: is documind-ingest deployed?"
print("revision 2 :", {k: line.get(k) for k in ("event", "chunks", "reused", "embedded", "retired", "effective_from")})
if line.get("reused") is None:
    print("the worker on the lane predates the carry-over (12 September) - restoring revision 1; Cloud Shell: "
          "make build deploy-services SERVICES_lean=ingest SCRIPTS_lean=commands/lesson-12.5.sh")
else:
    assert line["reused"] > line["embedded"], "a one-clause edit reuses most of the handbook (281 of 283 by section)"
    time.sleep(10)
    rc = scoped_gate()
    print(f"exit {rc}:", "RED, as it must be - lk-06 and vr-01 still expect 60 days; a real re-issue moves them in the same commit"
          if rc else "green?! the rows did not turn red - is RETRIEVAL_CURRENT_ONLY on the API, and did the worker swap?")
since = datetime.datetime.now(datetime.timezone.utc)
upload(V1)
back = wait_for(("ingest_reactivated", "ingest_ok"), since)
print("the undo   :", {k: (back or {}).get(k) for k in ("event", "chunks", "reused", "embedded", "retired")})
if line.get("reused") is not None:
    time.sleep(10)
    rc = scoped_gate()
    print(f"exit {rc}:", "green - the lane ends where it began" if rc == 0 else "still red: read the rows above")


## Demo-day replay checklist

Every card in the lesson has a replay. The ones the lane survives, in the order the session runs them:

| When | What | Where | Red, then green |
|---|---|---|---|
| 0:00 | the scoreboard as it stands | `make eval-live PROJECT=... 2>&1 \| tee ~/eval-0.log` | five rates, the miss list |
| 0:08 | who is asking | Setup, Cell 1, Cell 2 above | 200 / 401 / 403 / 422 |
| 0:15 | F1 twenty services per call | `gcloud services enable` with 32 names, then `make apis` | INVALID_ARGUMENT, then enabled |
| 0:18 | F8 Make kept the blanks | `make build PROJECT=... PROFILE="lean "` | ERROR: names no services |
| 0:20 | F10 / F11 identity | mint as `documind-api-sa`; mint without `--include-email` | PERMISSION_DENIED; 401 |
| 0:26 | F13 a smoke test that can say no | `DOCUMIND_SMOKE_QUESTION="Which file types are supported?" make smoke ...` | [FAIL], then 3/3 |
| 0:32 | F6 / F7 Firestore | `make adopt-firestore`; `make drift`; comment `__name__`, `make plan`, restore | already managed; no drift; must be replaced; no drift |
| 0:42 | the corpus as it went | Cell 6 above; cards F16-F19 on screen | statuses, counts, an empty DLQ |
| 0:52 | the loop, round 1 | Cells 4 and 5; one change; API-only redeploy; `make eval-live` | answerable_rate moves, the other four hold |
| 1:20 | round 2, or the retrospective | Cell 7 | the ledger |
| 1:22 | a document changes | Cell 10 above; `make reindex FILE=evals/demo/hr_policy_2026_v2.md NAME=hr_policy_2026.md TENANT=acme` | reused / embedded / retired on the line; the scoped gate red on lk-06 and vr-01; the undo; green |
| 1:24 | the eight knobs, measured | `make ablate PROJECT=... ABLATE_ARGS="--limit 5"` live; Cell 9's table from the 8th | reranker lift, depth not the knob, hybrid equals the lane |
| 1:28 | hold the line, then the bill | `make down PROJECT=...` (or `MIN_INSTANCES=0`) | no cost bleed |

The gate's own findings (F20-F23) replay on the product's rollback: every API fix is a Cloud Run revision still deployed with no traffic, tagged in git (`demo/state-0-refusals-disguised` ... `demo/state-4-chunk-ids-tenant-scoped`). Flip traffic to the state-0 revision, run the gate (fourteen refusals with nothing cited), flip back (`--to-latest`), run it again. The state-2 gate from git against today's API brings the spelling misses back with no deploy.

Not replayed, told from the cards with the exact first line on screen: F3 (a fresh sixty-resource apply), F4 and F5 (one-time API refusals the kit no longer sends), F16-F18 (a corpus load: Document AI, DLP, embeddings, money and twenty minutes).

The API-only redeploy, for the loop:

```bash
make build deploy-services PROJECT=documind-ai-YOUR-ID SERVICES_lean=api SCRIPTS_lean=commands/lesson-12.2.sh
```


## Done — Module 4, measured

- the five thresholds, and why one of them is 1.00
- two identities, three flags, one grant
- a miss list that names the place, and forensics that name the rung
- nineteen findings from the first live run, each with a replay or a story

Six runs took the gate from 55.8% to 97.7%: a parse failure disguised as a refusal (F20), a quote longer than the contract (F21), a gate that compared spellings (F22), a chunk id without a tenant (F23). No threshold moved.

Next: **10.4** scores the same golden set with Vertex AI's judged metrics; **12.7** runs this gate as the verify job of a keyless pipeline; **13.2** asks for all of this over your own corpus.